In [ ]:
# [fugu-core]
import os
from dataclasses import dataclass
from typing import Any

# Vendor -> env var that gates whether that expert slot is available this run.
PROVIDER_ENV = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "gemini": "GEMINI_API_KEY", "xai": "XAI_API_KEY", "deepseek": "DEEPSEEK_API_KEY"}

SYNTHESIZER_PROMPT = """Combine the original request and labeled expert outputs into one complete answer. Reconcile conflicts, remove redundancy, and do not mention the orchestration machinery."""
VERIFIER_PROMPT = """Check the draft against the original request, fix errors or gaps, and return only the final answer."""
EXPERT_PROMPT = """You are a {specialty} expert inside Fugu. Solve the request completely."""

@dataclass
class PoolModel:
    key: str; provider: str; agent: Any

class Fugu:
    def __init__(self, pool, synthesizer, verifier=None, *, ultra=False, exclude=()):
        self.pool, self.synthesizer, self.verifier, self.ultra = [m for m in pool if m.key not in exclude], synthesizer, verifier, ultra
    def run(self, prompt):
        # One-call facade: the caller never sees the team assembly.
        return self.orchestrate(prompt)
    def orchestrate(self, prompt):
        # Keep only experts whose provider key is present this run.
        available = [m for m in self.pool if os.getenv(PROVIDER_ENV.get(m.provider, ""))]
        if not available:
            raise RuntimeError("Fugu: no available experts in pool")
        # Every expert tackles the full request; label each output by specialty.
        outputs = [f"[{m.key}] {m.agent.run(prompt).content}" for m in available]
        # Merge the team's outputs into one authoritative answer.
        answer = self.synthesizer.run(f"Original request:\n{prompt}\n\nExpert outputs:\n" + "\n\n".join(outputs)).content
        # Fugu Ultra: one critique/finalize pass over the draft answer.
        if self.ultra and self.verifier:
            return self.verifier.run(f"Request:\n{prompt}\n\nDraft answer:\n{answer}").content
        return answer

# Live wiring + usage: runs only in a notebook (__main__), skipped by the offline test harness.
if __name__ == "__main__":
    from vidbyte import BaseAgent
    def agent(name, prompt, **kw): return BaseAgent(name=name, system_prompt=prompt, **kw)
    synthesizer, verifier = agent("fugu-synthesizer", SYNTHESIZER_PROMPT), agent("fugu-verifier", VERIFIER_PROMPT)
    def expert(key, provider, specialty): return PoolModel(key, provider, agent(f"expert-{key}", EXPERT_PROMPT.format(specialty=specialty)))
    pool = [expert("reasoning", "anthropic", "deep reasoning"), expert("coding", "openai", "software engineering"), expert("long_context", "gemini", "long-context synthesis"), expert("fast", "deepseek", "fast drafting")]
    fugu = Fugu(pool, synthesizer, verifier=verifier)
    print(fugu.run("Compare why Concorde retired with what a modern supersonic revival must solve."))
